In [0]:
from pyspark.sql.types import *
from pyspark.sql import   functions as F
from delta import DeltaTable

In [0]:
pip install  pandas

In [0]:
%sql
-- CREATING A  CATLOG IF NOT EXISTS 
USE CATALOG nyctaxi;
CREATE DATABASE IF NOT EXISTS nyctaxi.testDB; 
CREATE VOLUME IF NOT EXISTS nyctaxi.testDB.inboundData;
CREATE TABLE IF NOT EXISTS nyctaxi.testDB.employees
(
   employeeId  BIGINT GENERATED ALWAYS AS IDENTITY,
   firstName varchar(20), 
   gender  varchar(20),
   startdate date,
   lastLogin VARCHAR(20),
   salary   long , 
   bonus     DECIMAL(34,2),
   seniorManager VARCHAR(20),
   team varchar(20)
)

In [0]:
_userSchema = StructType(
                         [
                              StructField("firstName" , StringType() ),
                              StructField("gender" , StringType() ),
                              StructField("startdate", StringType()),
                              StructField("lastLogin",StringType()),
                              StructField("salary" , LongType()),
                              StructField("bonus" , DecimalType(34,2)),
                              StructField("seniorManager" , StringType()),
                              StructField("team",StringType())
                      ]
)

In [0]:
# Reading the data from the  volume and loading it into the delta table 
employeesDF = spark.read.csv(path='/Volumes/nyctaxi/testdb/inbounddata/employees.csv',
                             header= True,
                             schema=_userSchema,
                             sep = ',')
employeesDF = employeesDF.withColumn("startdate",F.to_date(F.col("startdate"),'M/d/yyyy'))
employeesDF = employeesDF.fillna({"Gender":"Unkown",
                    "Team":"Undeclared"})


In [0]:
# understaning the merge schema 
employeesDF.write.format("delta").mode("overWrite").option("mergeSchema",True).saveAsTable("nyctaxi.testDB.employees")
#  aggregating the data  to see how it will be shown in the UI 
employeesDF.groupBy(F.col("Team")).agg(F.sum("Salary").alias("groupedSalary")).show()